# 01 — Data Loading and Text Preprocessing
Based on the report appendix: cleaning, lowercasing, tokenisation, stop-word removal, stemming/lemmatisation. Upload a CSV containing `tweet_text` and `label`. The dataset itself is not included.

In [ ]:
!pip -q install nltk wordcloud seaborn scikit-learn
import re, numpy as np, pandas as pd, nltk, matplotlib.pyplot as plt, seaborn as sns
from google.colab import files
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer, WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from scipy.sparse import hstack
for r in ["punkt","punkt_tab","stopwords","wordnet","omw-1.4"]: nltk.download(r, quiet=True)

def clean_tweet(tweet):
    if pd.isna(tweet): return ""
    tweet=str(tweet)
    tweet=re.sub(r"https?://\S+|www\.\S+", " url ", tweet, flags=re.I)
    tweet=re.sub(r"@\w+", " user ", tweet)
    tweet=re.sub(r"#(\w+)", r"\1", tweet)
    tweet=re.sub(r"[^\w\s]", " ", tweet)
    return re.sub(r"\s+", " ", tweet).strip().lower()

def preprocess_text(text, remove_stopwords=True, stem=False, lemmatize=True, custom_stopwords=None):
    tokens=word_tokenize(clean_tweet(text))
    stops=set(stopwords.words("english")) if remove_stopwords else set()
    stops.update(custom_stopwords or [])
    tokens=[w for w in tokens if w not in stops and w.strip()]
    if stem:
        stemmer=PorterStemmer(); tokens=[stemmer.stem(w) for w in tokens]
    if lemmatize:
        lemma=WordNetLemmatizer(); tokens=[lemma.lemmatize(w) for w in tokens]
    return " ".join(tokens)

def load_csv():
    uploaded=files.upload()
    name=next((n for n in uploaded if n.lower().endswith(".csv")),None)
    if not name: raise ValueError("Upload a CSV file.")
    data=pd.read_csv(name)
    if not {"tweet_text","label"}.issubset(data.columns):
        raise ValueError("CSV must contain columns named tweet_text and label. Rename your columns if needed.")
    data=data.dropna(subset=["label"]).copy()
    data["tweet_text"]=data["tweet_text"].fillna("").astype(str)
    return data

In [ ]:
df=load_csv()
display(df.head())
print("Shape:",df.shape," Labels:",df["label"].nunique())

## Clean and preprocess text

In [ ]:
CUSTOM_STOPWORDS={"url","user","rt","mkr"}
df["tweet_text_clean"]=df["tweet_text"].apply(clean_tweet)
df["tweet_text_processed"]=df["tweet_text"].apply(lambda x: preprocess_text(x, custom_stopwords=CUSTOM_STOPWORDS))
display(df[["tweet_text","tweet_text_processed","label"]].head(10))
df[["tweet_text","tweet_text_processed","label"]].to_csv("processed_dataset.csv",index=False)
files.download("processed_dataset.csv")